In [0]:
# ============================================================
# FOOD DELIVERY PERFORMANCE & CUSTOMER ANALYTICS PIPELINE
# Gold Layer - Business Analytics
# Roll Number: 23052151
# ============================================================

print("Food Delivery Analytics Pipeline")
print("Roll Number: 23052151")
print("Layer: Gold")

Food Delivery Analytics Pipeline
Roll Number: 23052151
Layer: Gold


In [0]:
# ============================================================
# PATHS
# ============================================================

SILVER_PATH = "/Volumes/workspace/capstone_23052151/silver_data"
GOLD_PATH = "/Volumes/workspace/capstone_23052151/gold_data"

print("Silver path:", SILVER_PATH)
print("Gold path:", GOLD_PATH)

Silver path: /Volumes/workspace/capstone_23052151/silver_data
Gold path: /Volumes/workspace/capstone_23052151/gold_data


In [0]:
# ============================================================
# IMPORTS
# ============================================================

from pyspark.sql.functions import (
    col,
    sum,
    avg,
    count,
    countDistinct,
    round,
    when,
    month,
    year,
    date_format,
    coalesce,
    lit
)

print("Functions imported successfully.")

Functions imported successfully.


In [0]:
# ============================================================
# READ SILVER DATA
# ============================================================

customers_silver = spark.read.format("delta").load(
    f"{SILVER_PATH}/customers"
)

restaurants_silver = spark.read.format("delta").load(
    f"{SILVER_PATH}/restaurants"
)

menu_items_silver = spark.read.format("delta").load(
    f"{SILVER_PATH}/menu_items"
)

orders_silver = spark.read.format("delta").load(
    f"{SILVER_PATH}/orders"
)

order_items_silver = spark.read.format("delta").load(
    f"{SILVER_PATH}/order_items"
)

delivery_silver = spark.read.format("delta").load(
    f"{SILVER_PATH}/delivery_details"
)

reviews_silver = spark.read.format("delta").load(
    f"{SILVER_PATH}/reviews"
)

delivery_performance = spark.read.format("delta").load(
    f"{SILVER_PATH}/delivery_performance"
)

print("All Silver datasets loaded successfully.")

All Silver datasets loaded successfully.


In [0]:
# ============================================================
# GOLD 1 - RESTAURANT PERFORMANCE
# ============================================================

# Average customer rating for each restaurant
restaurant_ratings = (
    orders_silver
    .join(
        reviews_silver.select(
            "order_id",
            "rating"
        ),
        on="order_id",
        how="left"
    )
    .groupBy("restaurant_id")
    .agg(
        round(avg("rating"), 2).alias("average_customer_rating")
    )
)

# Delivery performance for each restaurant
restaurant_delivery = (
    delivery_performance
    .filter(
        col("actual_delivery_time").isNotNull()
    )
    .groupBy(
        "restaurant_id",
        "restaurant_name",
        "city",
        "cuisine_type"
    )
    .agg(
        countDistinct("order_id").alias("delivered_orders"),
        sum("is_late").alias("late_orders")
    )
    .withColumn(
        "late_delivery_rate",
        round(
            col("late_orders") / col("delivered_orders") * 100,
            2
        )
    )
)

restaurant_performance = (
    restaurant_delivery
    .join(
        restaurant_ratings,
        on="restaurant_id",
        how="left"
    )
    .select(
        "restaurant_id",
        "restaurant_name",
        "city",
        "cuisine_type",
        "delivered_orders",
        "late_orders",
        "late_delivery_rate",
        "average_customer_rating"
    )
)

display(
    restaurant_performance
    .orderBy(
        col("late_delivery_rate").desc()
    )
    .limit(20)
)

restaurant_id,restaurant_name,city,cuisine_type,delivered_orders,late_orders,late_delivery_rate,average_customer_rating
R0235,Restaurant_0235,Bengaluru,Continental,211,190,90.05,4.06
R0230,Restaurant_0230,Mumbai,Chinese,173,155,89.6,3.96
R0185,Restaurant_0185,Pune,South Indian,170,152,89.41,4.12
R0291,Restaurant_0291,Chennai,Bengali,189,168,88.89,4.02
R0350,Restaurant_0350,Mumbai,Italian,171,152,88.89,4.03
R0113,Restaurant_0113,Kolkata,South Indian,196,174,88.78,4.07
R0170,Restaurant_0170,Bengaluru,South Indian,202,179,88.61,4.11
R0194,Restaurant_0194,Pune,South Indian,190,168,88.42,4.09
R0280,Restaurant_0280,Hyderabad,South Indian,197,174,88.32,4.24
R0384,Restaurant_0384,Hyderabad,Italian,202,178,88.12,4.06


In [0]:
# ============================================================
# GOLD 2 - CUSTOMER SEGMENT PERFORMANCE
# ============================================================

# Calculate revenue at order-item level
order_revenue = (
    order_items_silver
    .groupBy("order_id")
    .agg(
        round(
            sum(
                coalesce(
                    col("item_revenue"),
                    col("quantity") * col("unit_price")
                )
            ),
            2
        ).alias("order_revenue")
    )
)

# Join orders with customers and revenue
customer_orders = (
    orders_silver
    .join(
        customers_silver.select(
            "customer_id",
            "customer_segment"
        ),
        on="customer_id",
        how="left"
    )
    .join(
        order_revenue,
        on="order_id",
        how="left"
    )
    .filter(
        ~col("order_status").isin(
            "Cancelled",
            "Canceled"
        )
    )
    .withColumn(
        "order_month",
        date_format(
            col("order_datetime"),
            "yyyy-MM"
        )
    )
)

customer_segment_performance = (
    customer_orders
    .groupBy(
        "order_month",
        "customer_segment"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        round(
            sum("order_revenue"),
            2
        ).alias("total_revenue")
    )
    .withColumn(
        "average_order_value",
        round(
            col("total_revenue") / col("total_orders"),
            2
        )
    )
    .orderBy(
        "order_month",
        col("total_revenue").desc()
    )
)

display(customer_segment_performance)

order_month,customer_segment,total_orders,unique_customers,total_revenue,average_order_value
2024-01,Regular,3343,2721,5342898.67,1598.23
2024-01,Frequent,2176,1788,3517864.99,1616.67
2024-01,Budget,2135,1723,3448578.84,1615.26
2024-01,Premium,881,709,1460574.92,1657.86
2024-02,Regular,3260,2658,5145723.5,1578.44
2024-02,Frequent,2044,1675,3293714.16,1611.41
2024-02,Budget,1942,1612,3176230.36,1635.55
2024-02,Premium,804,655,1324060.21,1646.84
2024-03,Regular,3421,2741,5587459.68,1633.28
2024-03,Frequent,2199,1803,3544978.64,1612.09


In [0]:
# ============================================================
# GOLD 3 - CATEGORY PERFORMANCE
# ============================================================

# Revenue by category
category_revenue = (
    order_items_silver
    .groupBy("category")
    .agg(
        round(
            sum(
                coalesce(
                    col("item_revenue"),
                    col("quantity") * col("unit_price")
                )
            ),
            2
        ).alias("total_revenue"),
        sum("quantity").alias("items_sold")
    )
)

# Get order-level ratings first
order_ratings = (
    reviews_silver
    .filter(
        col("rating").isNotNull()
    )
    .groupBy("order_id")
    .agg(
        avg("rating").alias("order_rating")
    )
)

# Connect categories to order ratings
category_ratings = (
    order_items_silver
    .join(
        order_ratings,
        on="order_id",
        how="inner"
    )
    .groupBy("category")
    .agg(
        round(
            avg("order_rating"),
            2
        ).alias("average_customer_rating")
    )
)

category_performance = (
    category_revenue
    .join(
        category_ratings,
        on="category",
        how="left"
    )
    .select(
        "category",
        "items_sold",
        "total_revenue",
        "average_customer_rating"
    )
    .orderBy(
        col("total_revenue").desc()
    )
)

display(category_performance)

category,items_sold,total_revenue,average_customer_rating
Desserts,38135,1.871402937E7,4.09
Pizza,37263,1.852863316E7,4.08
Bengali,36624,1.829679861E7,4.07
Beverages,38178,1.821254667E7,4.1
Chinese,37133,1.814617767E7,4.07
Burger,36608,1.78408417E7,4.08
Healthy,35913,1.748655328E7,4.08
Biryani,35426,1.748176014E7,4.1
South Indian,35207,1.682283957E7,4.08


In [0]:
# ============================================================
# GOLD QUALITY CHECKS
# ============================================================

print("========== GOLD TABLE COUNTS ==========")

print(
    "Restaurant Performance:",
    restaurant_performance.count()
)

print(
    "Customer Segment Performance:",
    customer_segment_performance.count()
)

print(
    "Category Performance:",
    category_performance.count()
)

print("\n========== NULL CHECKS ==========")

print(
    "Restaurant IDs missing:",
    restaurant_performance
    .filter(col("restaurant_id").isNull())
    .count()
)

print(
    "Customer segments missing:",
    customer_segment_performance
    .filter(col("customer_segment").isNull())
    .count()
)

print(
    "Categories missing:",
    category_performance
    .filter(col("category").isNull())
    .count()
)

========== GOLD TABLE COUNTS ==========
Restaurant Performance: 500
Customer Segment Performance: 48
Category Performance: 9

========== NULL CHECKS ==========
Restaurant IDs missing: 0
Customer segments missing: 0
Categories missing: 0


In [0]:
# ============================================================
# WRITE GOLD DATA
# ============================================================

restaurant_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/restaurant_performance")

customer_segment_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/customer_segment_performance")

category_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/category_performance")

print("All Gold tables written successfully.")

All Gold tables written successfully.


In [0]:
# ============================================================
# VERIFY GOLD LAYER
# ============================================================

display(dbutils.fs.ls(GOLD_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/capstone_23052151/gold_data/category_performance/,category_performance/,0,1790658714263
dbfs:/Volumes/workspace/capstone_23052151/gold_data/customer_segment_performance/,customer_segment_performance/,0,1790658714263
dbfs:/Volumes/workspace/capstone_23052151/gold_data/restaurant_performance/,restaurant_performance/,0,1790658714263


In [0]:
# ============================================================
# FINAL BUSINESS ANSWERS
# ============================================================

from pyspark.sql import functions as F

# Reload the three Gold tables
restaurant = spark.read.format("delta").load(
    "/Volumes/workspace/capstone_23052151/gold_data/restaurant_performance"
)

customer = spark.read.format("delta").load(
    "/Volumes/workspace/capstone_23052151/gold_data/customer_segment_performance"
)

category = spark.read.format("delta").load(
    "/Volumes/workspace/capstone_23052151/gold_data/category_performance"
)

# ------------------------------------------------------------
# Q1: Restaurants with highest late-delivery rates
# ------------------------------------------------------------

q1 = (
    restaurant
    .filter(F.col("delivered_orders") > 0)
    .select(
        "restaurant_id",
        "restaurant_name",
        "city",
        "cuisine_type",
        "delivered_orders",
        "late_orders",
        F.col("late_delivery_rate").alias("late_delivery_percentage"),
        "average_customer_rating"
    )
    .orderBy(F.col("late_delivery_percentage").desc())
)

print("QUESTION 1 — TOP RESTAURANTS BY LATE DELIVERY RATE")
display(q1.limit(10))

# ------------------------------------------------------------
# Q2: Customer segment revenue and ordering behaviour
# ------------------------------------------------------------

q2 = (
    customer
    .groupBy("customer_segment")
    .agg(
        F.sum("total_orders").alias("total_orders"),
        F.sum("unique_customers").alias("unique_customers"),
        F.sum("total_revenue").alias("total_revenue"),
        F.avg("average_order_value").alias("average_order_value")
    )
    .orderBy(F.col("total_revenue").desc())
)

print("QUESTION 2 — CUSTOMER SEGMENT PERFORMANCE")
display(q2)


# Monthly segment trend
q2_trend = (
    customer
    .select(
        "order_month",
        "customer_segment",
        "total_orders",
        "total_revenue",
        "average_order_value"
    )
    .orderBy("order_month", F.col("total_revenue").desc())
)

print("QUESTION 2 — MONTHLY TREND")
display(q2_trend)


# ------------------------------------------------------------
# Q3: Category revenue and ratings
# ------------------------------------------------------------

q3 = (
    category
    .select(
        "category",
        "items_sold",
        "total_revenue",
        "average_customer_rating"
    )
    .orderBy(F.col("total_revenue").desc())
)

print("QUESTION 3 — CATEGORY PERFORMANCE")
display(q3.limit(10))

QUESTION 1 — TOP RESTAURANTS BY LATE DELIVERY RATE


restaurant_id,restaurant_name,city,cuisine_type,delivered_orders,late_orders,late_delivery_percentage,average_customer_rating
R0235,Restaurant_0235,Bengaluru,Continental,211,190,90.05,4.06
R0230,Restaurant_0230,Mumbai,Chinese,173,155,89.6,3.96
R0185,Restaurant_0185,Pune,South Indian,170,152,89.41,4.12
R0291,Restaurant_0291,Chennai,Bengali,189,168,88.89,4.02
R0350,Restaurant_0350,Mumbai,Italian,171,152,88.89,4.03
R0113,Restaurant_0113,Kolkata,South Indian,196,174,88.78,4.07
R0170,Restaurant_0170,Bengaluru,South Indian,202,179,88.61,4.11
R0194,Restaurant_0194,Pune,South Indian,190,168,88.42,4.09
R0280,Restaurant_0280,Hyderabad,South Indian,197,174,88.32,4.24
R0384,Restaurant_0384,Hyderabad,Italian,202,178,88.12,4.06


QUESTION 2 — CUSTOMER SEGMENT PERFORMANCE


customer_segment,total_orders,unique_customers,total_revenue,average_order_value
Regular,39598,32406,6.380390201E7,1611.2891666666665
Frequent,25819,21138,4.185919691E7,1621.1091666666669
Budget,24619,20156,3.978734547E7,1616.1241666666667
Premium,9964,8116,1.607973578E7,1613.565833333333


QUESTION 2 — MONTHLY TREND


order_month,customer_segment,total_orders,total_revenue,average_order_value
2024-01,Regular,3343,5342898.67,1598.23
2024-01,Frequent,2176,3517864.99,1616.67
2024-01,Budget,2135,3448578.84,1615.26
2024-01,Premium,881,1460574.92,1657.86
2024-02,Regular,3260,5145723.5,1578.44
2024-02,Frequent,2044,3293714.16,1611.41
2024-02,Budget,1942,3176230.36,1635.55
2024-02,Premium,804,1324060.21,1646.84
2024-03,Regular,3421,5587459.68,1633.28
2024-03,Frequent,2199,3544978.64,1612.09


QUESTION 3 — CATEGORY PERFORMANCE


category,items_sold,total_revenue,average_customer_rating
Desserts,38135,1.871402937E7,4.09
Pizza,37263,1.852863316E7,4.08
Bengali,36624,1.829679861E7,4.07
Beverages,38178,1.821254667E7,4.1
Chinese,37133,1.814617767E7,4.07
Burger,36608,1.78408417E7,4.08
Healthy,35913,1.748655328E7,4.08
Biryani,35426,1.748176014E7,4.1
South Indian,35207,1.682283957E7,4.08
